# Personal AI on Google Colab

**Qwen3-4B-Instruct-2507 · free model weights · T4 GPU · password-protected chat app**

This notebook is deliberately kept simple: load one stable open-weight language model in 4-bit mode, run a quick test, then launch a small web chat app.

### Before you start
1. In Colab, choose **Runtime → Change runtime type → T4 GPU** (or another available GPU).
2. Run each cell from top to bottom.
3. The first model download can take several minutes and downloads multiple GB.
4. The final cell prints a temporary web link. Keep this runtime running while you use the app.

**Limitations:** Colab GPU time is not guaranteed or unlimited. Sessions disconnect. The public Gradio tunnel is protected by the username/password you enter in the last cell, but treat the link like a password and don't enter sensitive data. This starter is a chatbot, not yet an autonomous agent: it has no web-search, shell, Google-account, or unrestricted file tools.

## 1. Check your runtime

If this cell says there is no CUDA GPU, stop here and select a GPU in **Runtime → Change runtime type**.

In [ ]:
import sys, torch
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No GPU allocated. In Colab, choose Runtime → Change runtime type → T4 GPU, then rerun this cell.")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
!nvidia-smi

## 2. Install only the app dependencies

This cell does not reinstall PyTorch. It pins Transformers to the 4.x series to avoid unrelated major-version changes and uses Gradio 5.x for a stable chat interface.

In [ ]:
%pip -q install "transformers>=4.51.0,<5.0" "accelerate>=1.2.0" "bitsandbytes>=0.45.0" "gradio>=5.0,<6.0"

## 3. Download and load Qwen3-4B in 4-bit

The model is published by Qwen under Apache 2.0. This notebook uses the official `Qwen/Qwen3-4B-Instruct-2507` checkpoint and quantizes its linear layers at load time to reduce VRAM usage.

In [ ]:
import gc, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU not detected. Select a GPU runtime before loading the model.")

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print(f"Loading {MODEL_ID}. The first run downloads the model files...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)
model.eval()
print("Model loaded.")
print("GPU:", torch.cuda.get_device_name(0))
print("Allocated VRAM GiB:", round(torch.cuda.memory_allocated(0) / 1024**3, 2))

## 4. Run a smoke test

If this works and prints a response, the model is loaded before we start the web app. If it fails, copy the full error output for troubleshooting.

In [ ]:
def _generate(messages, max_new_tokens=128, temperature=0.3):
    # Keep the prompt modest for Colab's limited GPU memory.
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    generation_args = {
        "max_new_tokens": int(max_new_tokens),
        "do_sample": float(temperature) > 0,
        "repetition_penalty": 1.05,
        "pad_token_id": tokenizer.eos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }
    if float(temperature) > 0:
        generation_args["temperature"] = float(temperature)
        generation_args["top_p"] = 0.9

    with torch.inference_mode():
        output = model.generate(**inputs, **generation_args)
    new_tokens = output[0, inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print(_generate([
    {"role": "system", "content": "You are a helpful, honest assistant. Be concise."},
    {"role": "user", "content": "Reply with one sentence: the personal AI model is running."},
], max_new_tokens=64, temperature=0.0))

## 5. Create your private-feeling chat app

The model runs on the Colab GPU. Gradio supplies the chat UI. For safety, the app has no shell, web browser, account integrations, or arbitrary tool execution.

In [ ]:
import gradio as gr
import getpass

SYSTEM_PROMPT = """You are a personal AI assistant. Be helpful, clear, practical, and honest about uncertainty. Do not claim to browse the live web, access the user's computer, remember previous sessions, or perform external actions unless a tool explicitly provides that ability. Ask a focused question only when missing information prevents a useful answer. Prefer actionable steps and explain important risks."""

def chat(message, history, max_new_tokens=384, temperature=0.4):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    # Keep only recent conversational turns to limit memory usage.
    for item in (history or [])[-10:]:
        if isinstance(item, dict) and item.get("role") in ("user", "assistant"):
            content = item.get("content", "")
            if isinstance(content, str) and content.strip():
                messages.append({"role": item["role"], "content": content})
    messages.append({"role": "user", "content": str(message)})
    answer = _generate(messages, max_new_tokens=int(max_new_tokens), temperature=float(temperature))
    return answer or "I couldn't generate a text response. Try a shorter prompt."

APP_USERNAME = input("Choose a username for the temporary app: ").strip()
if not APP_USERNAME:
    raise ValueError("Username cannot be blank.")
APP_PASSWORD = getpass.getpass("Choose a unique password (at least 14 characters): ")
if len(APP_PASSWORD) < 14:
    raise ValueError("Use a unique password at least 14 characters long, then rerun this cell.")

demo = gr.ChatInterface(
    fn=chat,
    type="messages",
    title="Personal AI",
    description="Qwen3-4B running on your current Colab GPU. The app stops when the runtime stops.",
    additional_inputs=[
        gr.Slider(64, 768, value=384, step=64, label="Maximum new tokens"),
        gr.Slider(0.0, 1.0, value=0.4, step=0.1, label="Creativity (temperature)"),
    ],
    examples=[
        "Explain how local AI models work.",
        "Help me plan a small open-source software project.",
        "Write a Python script that renames files safely, with a dry-run mode.",
    ],
)

demo.launch(
    share=True,
    auth=(APP_USERNAME, APP_PASSWORD),
    show_error=True,
    debug=True,
)


## Troubleshooting

- **No GPU / `CUDA available: False`:** Runtime → Change runtime type → T4 GPU, then rerun from cell 1.
- **Out of memory:** Runtime → Disconnect and delete runtime, reconnect with a GPU, then rerun. Keep token limits modest.
- **Transformers / bitsandbytes import error:** Restart the runtime after the install cell, then rerun cells from the top. Do not manually upgrade PyTorch.
- **The model download failed:** Rerun the model-loading cell; check Colab's internet access and Hugging Face status.
- **The share link cannot be reached:** The runtime may have disconnected. Rerun the launch cell; a new temporary link may be generated.

## Important limitations

- The model weights and inference are open-weight/local to this Colab runtime, but the machine itself is hosted by Google.
- Each fresh runtime may need to download the model again.
- Gradio share links are temporary and reachable from the internet. Use the password, don't share it, and don't enter sensitive information.
- This is not an always-on server. A persistent 24/7 deployment usually costs money.